# Minggu 6 · Sharpening dan Edge Enhancement
## Sub-CPMK 6 | Notebook praktikum terpandu

**Empat teknik:** Sobel, Laplacian, Unsharp Masking / High-Boost, dan Canny. Durasi perkuliahan 150 menit: teori 50, demonstrasi 50, praktik 50. Praktik wajib mengulang data serta parameter yang dijelaskan pengajar. Eksperimen tambahan tersedia setelah bagian F.

**Cara memakai Colab:** unggah berkas `.ipynb` ke [Google Colab](https://colab.research.google.com/), pilih runtime CPU, lalu **Runtime → Run all**. Dua citra kecil sudah tertanam di notebook; tidak perlu mengunduh seluruh dataset atau memasang Drive. Untuk praktik, baca penjelasan lalu jalankan sel A1 sampai F2 berurutan. Waktu 50 menit mencakup membaca hasil dan menulis pengamatan.

**Bukti belajar:** Gx/Gy/magnitude, Laplacian bertanda dan citra tajam, USM k=1 dan k=2 pada ROI sama, dua Canny pada masukan sama, serta alasan memilih hasil. Jumlah piksel tepi dan persentase clipping hanya deskripsi keluaran, bukan skor kualitas.

**Nama:** ...  |  **NIM:** ...  |  **Kelas:** ...

Isi identitas pada sel teks ini sebelum menyimpan hasil praktikum.

## A · Persiapan dan dua data nyata (5 menit)
**Misi 1:** mengamati batas tulang pada radiograf siku nyata, bukan melakukan diagnosis. **Misi 2:** mengamati batas jalan, atap, dan tekstur pada citra satelit nyata, bukan menghasilkan jaringan jalan GIS.

1. **FracAtlas, v7:** `IMG0000025.jpg`, Abedeen dkk. Radiograf asli 2880×2304 diperkecil menjadi 800×640, grayscale, interpolasi area. Tidak ditambah retakan atau diburamkan secara sintetis. [Dataset resmi](https://doi.org/10.6084/m9.figshare.22363012.v7) · [paper](https://www.nature.com/articles/s41597-023-02432-4) · **CC BY 4.0**.
2. **SpaceNet 3:** `RGB-PanSharpen_AOI_5_Khartoum_img194.tif`, SpaceNet Partners. Satelit pansharpened RGB, 1300×1300 → 650×650. Setiap kanal valid diregangkan pada persentil 2–98, lalu dikonversi 8-bit dan diperkecil dengan interpolasi area. PNG ini adaptasi untuk tampilan; metadata geospasial tidak disertakan. [Dataset resmi](https://www.spacenet.ai/spacenet-roads-dataset/) · **CC BY-SA 4.0**; adaptasi gambar tetap CC BY-SA 4.0.

Parameter dalam notebook berlaku untuk versi citra di atas. Resolusi dan peregangan intensitas memengaruhi gradien serta ambang; nilai Canny ini tidak universal.

In [ ]:
# A1 : impor. Colab biasanya sudah menyediakan paket ini.
import importlib, subprocess, sys
packages = {"cv2": "opencv-python-headless", "numpy": "numpy", "matplotlib": "matplotlib"}
for module, package in packages.items():
    try:
        importlib.import_module(module)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
import cv2, numpy as np, matplotlib.pyplot as plt
import base64, hashlib, json
from pathlib import Path
print("Python", sys.version.split()[0], "| OpenCV", cv2.__version__, "| NumPy", np.__version__)
print("Runtime CPU cukup. Data siap dipakai tanpa unduhan eksternal.")

In [ ]:
# A2 : data tertanam; sel panjang ini cukup dijalankan sekali.
#@title A2: Muat dua citra nyata dan cek integritas
DATA_B64 = {'xray_fracatlas_IMG0000025.png': '...', 'satellite_spacenet3_Khartoum_img194.png': '...'}
EXPECTED_SHA256 = {'xray_fracatlas_IMG0000025.png': '7613319b6fdb56f35638c7b82c1a846dd089bbb4cb5006b5cd3c50a42f49dc7a', 'satellite_spacenet3_Khartoum_img194.png': '8180e50da298b102a34cc5d0fb53e2ff5d34d7f1211ae3102c9c7a528010b507'}
data_dir = Path("subcpmk6_data")
data_dir.mkdir(exist_ok=True)
for name, encoded in DATA_B64.items():
    raw = base64.b64decode(encoded)
    assert hashlib.sha256(raw).hexdigest() == EXPECTED_SHA256[name]
    (data_dir / name).write_bytes(raw)
xray_u8 = cv2.imread(str(data_dir / "xray_fracatlas_IMG0000025.png"), cv2.IMREAD_GRAYSCALE)
sat_bgr = cv2.imread(str(data_dir / "satellite_spacenet3_Khartoum_img194.png"), cv2.IMREAD_COLOR)
assert xray_u8 is not None and sat_bgr is not None
sat_rgb = cv2.cvtColor(sat_bgr, cv2.COLOR_BGR2RGB)
xray = xray_u8.astype(np.float64)
print("X-ray:", xray_u8.shape, xray_u8.dtype, "| Satelit:", sat_rgb.shape, sat_rgb.dtype)
print("SHA-256 dua citra sesuai. Urutan warna tampilan sudah RGB.")

In [ ]:
# A3 : satu kontrak parameter untuk semua perbandingan.
GAUSS = (5, 5)
SIGMA = 1.0
LAMBDA = 0.25
K_VALUES = (1.0, 2.0)
THRESHOLDS = ((50, 150), (100, 200))
XRAY_ROI = (370, 68, 560, 230)   # (x1, y1, x2, y2), batas tulang di siku
TISSUE_ROI = (650, 200, 770, 300) # jaringan nyata; SD di sini bukan noise murni
ROAD_ROI = (110, 160, 310, 360)
ROOF_ROI = (390, 250, 590, 450)
plt.rcParams.update({"font.size": 11, "figure.dpi": 110, "savefig.dpi": 140})

def crop(a, roi):
    x1, y1, x2, y2 = roi
    return a[y1:y2, x1:x2]

def grid(arrays, titles, cmaps=None, limits=None, ncols=None, figsize=None):
    ncols = ncols or len(arrays)
    nrows = int(np.ceil(len(arrays)/ncols))
    fig, axs = plt.subplots(nrows, ncols, figsize=figsize or (4*ncols, 3.5*nrows), squeeze=False)
    for i, (a, title) in enumerate(zip(arrays, titles)):
        ax = axs.ravel()[i]
        kwargs = {}
        if a.ndim == 2:
            kwargs["cmap"] = (cmaps[i] if cmaps else "gray")
            if limits and limits[i] is not None: kwargs.update(vmin=limits[i][0], vmax=limits[i][1])
        ax.imshow(a, **kwargs)
        ax.set_title(title, fontsize=12); ax.axis("off")
    for ax in axs.ravel()[len(arrays):]: ax.axis("off")
    fig.tight_layout(); plt.show()
    return None

def to_u8(a):
    return np.clip(a, 0, 255).astype(np.uint8)

def clipping_pct(a):
    return 100*np.count_nonzero((a<0)|(a>255))/a.size

print("Float 0..255 dipertahankan sampai rekonstruksi selesai. Clipping hanya di akhir.")

In [ ]:
# A4 : lokasi ROI konsisten pada semua metode.
fig, axs = plt.subplots(1, 2, figsize=(11, 4.6))
axs[0].imshow(xray_u8, cmap="gray", vmin=0, vmax=255)
axs[0].set_title("Radiograf siku nyata · FracAtlas IMG0000025")
axs[1].imshow(sat_rgb); axs[1].set_title("Satelit nyata · SpaceNet 3 Khartoum img194")
from matplotlib.patches import Rectangle
for ax, roi, color, label in [(axs[0],XRAY_ROI,"#ffbc42","batas tulang"), (axs[0],TISSUE_ROI,"#6ce5c0","jaringan"), (axs[1],ROAD_ROI,"#ffbc42","jalan"), (axs[1],ROOF_ROI,"#6ce5c0","atap/tekstur")]:
    x1,y1,x2,y2=roi
    ax.add_patch(Rectangle((x1,y1),x2-x1,y2-y1,fill=False,edgecolor=color,linewidth=2))
    ax.text(x1,y2+16,label,color=color,fontsize=10,bbox={"facecolor":"#172035","alpha":0.8,"pad":2})
for ax in axs: ax.axis("off")
fig.tight_layout(); plt.show()

## B · Sobel: perubahan menurut dua arah (8 menit)
Gx mengukur perubahan intensitas kiri–kanan, sehingga batas vertikal tampak kuat. Gy mengukur perubahan atas–bawah, sehingga batas horizontal tampak kuat. Magnitude memakai `hypot(Gx, Gy)`, arah memakai `atan2(Gy, Gx)`.

Respons turunan harus **bertanda**: gelap→terang dan terang→gelap memiliki tanda berlawanan. `CV_64F` menyimpan keduanya. Untuk panel magnitude dipakai batas tampilan tetap 0–1020, bukan normalisasi otomatis masing-masing hasil. Nilai float asli tetap disimpan.

In [ ]:
# B1 : kartu uji terkontrol: bentuk/tepi diketahui, bukan data medis sintetis.
card = np.full((256, 384), 40, dtype=np.uint8)
cv2.rectangle(card, (48, 40), (164, 218), 200, -1)
cv2.rectangle(card, (220, 100), (335, 150), 120, -1)
cv2.line(card, (230, 215), (340, 40), 220, 3)
fig, axs = plt.subplots(1, 2, figsize=(10, 3.7))
axs[0].imshow(card,cmap="gray",vmin=0,vmax=255);axs[0].axhline(128,color="#ff8b33",lw=1);axs[0].set_title("Kartu uji · garis pengambilan profil");axs[0].axis("off")
axs[1].plot(card[128],color="#6d48d7");axs[1].set_title("Profil intensitas baris 128");axs[1].set(xlabel="Kolom x (piksel)",ylabel="Intensitas",ylim=(0,255));axs[1].grid(alpha=.2)
fig.tight_layout();plt.show()

In [ ]:
# B2 : tiga keluaran berbeda: respons x, respons y, magnitude.
card_f = card.astype(np.float64)
gx_card = cv2.Sobel(card_f, cv2.CV_64F, 1, 0, ksize=3)
gy_card = cv2.Sobel(card_f, cv2.CV_64F, 0, 1, ksize=3)
mag_card = np.hypot(gx_card, gy_card)
angle_card = np.degrees(np.arctan2(gy_card, gx_card))
grid([gx_card, gy_card, mag_card], ["Gx: perubahan kiri–kanan", "Gy: perubahan atas–bawah", "Magnitude: hypot(Gx, Gy)"], cmaps=["coolwarm","coolwarm","gray"], limits=[(-1020,1020),(-1020,1020),(0,1020)], figsize=(12,3.5))
print("Rentang Gx:",gx_card.min(),gx_card.max(),"| Gy:",gy_card.min(),gy_card.max())
print("Sudut tidak ditafsirkan pada daerah dengan magnitude nol.")

In [ ]:
# B3 : operator yang sama pada radiograf nyata.
gx = cv2.Sobel(xray, cv2.CV_64F, 1, 0, ksize=3)
gy = cv2.Sobel(xray, cv2.CV_64F, 0, 1, ksize=3)
mag = np.hypot(gx, gy)
theta_deg = np.degrees(np.arctan2(gy, gx))
grid([crop(xray,XRAY_ROI),crop(gx,XRAY_ROI),crop(gy,XRAY_ROI),crop(mag,XRAY_ROI)], ["ROI masukan", "Gx bertanda", "Gy bertanda", "Magnitude"], cmaps=["gray","coolwarm","coolwarm","gray"],limits=[(0,255),(-1020,1020),(-1020,1020),(0,1020)],figsize=(12,3.5))
print("Magnitude maksimum (float):",round(float(mag.max()),2))

## C · Laplacian: turunan kedua dan rekonstruksi (9 menit)
Pada `ksize=1`, OpenCV memakai kernel pusat −4: `[[0,1,0],[1,-4,1],[0,1,0]]`. Jumlah kernel nol: area konstan di bagian interior memberi respons nol. Dengan konvensi tanda ini, sharpening adalah `g = I - lambda*L`.

Laplacian mentah bukan citra tajam. Rekonstruksi harus menggabungkannya dengan masukan. `abs(L)` hanya cocok untuk melihat besar respons; jangan memasukkannya ke formula rekonstruksi karena tanda hilang. Zero-crossing berarti perubahan tanda, bukan sekadar piksel bernilai nol.

In [ ]:
# C1 : contoh angka dan cek kernel.
LAPLACIAN_KERNEL = np.array([[0,1,0],[1,-4,1],[0,1,0]],dtype=np.float64)
assert LAPLACIAN_KERNEL.sum() == 0
f_1d = np.array([20,20,20,220,220,220],dtype=float)
first = np.diff(f_1d)
second = f_1d[:-2]-2*f_1d[1:-1]+f_1d[2:]
print("Intensitas:",f_1d)
print("Selisih pertama:",first,"| Selisih kedua pada interior:",second)
print("Kernel Laplacian ksize1:",LAPLACIAN_KERNEL,sep="\n")
print("Kernel sharpening lambda0.25: pusat2, empat tetangga−0.25; jumlah1.")

In [ ]:
# C2 : tanda, gain, dan clipping diperiksa secara terpisah.
lap = cv2.Laplacian(xray, cv2.CV_64F, ksize=1)
lap_sharp_float = xray - LAMBDA*lap
lap_sharp = to_u8(lap_sharp_float)
grid([crop(xray,XRAY_ROI),crop(lap,XRAY_ROI),crop(lap_sharp,XRAY_ROI)], ["Masukan: ROI sama", "Laplacian bertanda", "I − 0.25L"],cmaps=["gray","coolwarm","gray"],limits=[(0,255),(-100,100),(0,255)],figsize=(11,3.7))
print("Rentang Laplacian:",lap.min(),lap.max())
print("Clipping rekonstruksi (% piksel):",round(clipping_pct(lap_sharp_float),4))
print("Panel bertanda memakai skala tetap −100..100; float asli tidak diubah.")

## D · Unsharp Masking dan High-Boost (10 menit)
`blur = Gaussian(I)` → `mask = I - blur` → `g = I + k*mask`. Sesuai konvensi slide dosen, **k=1** adalah USM standar; **k=2** adalah high-boost. `cv2.addWeighted(I, 1+k, blur, -k, 0)` ekuivalen pada float. Citra uint8 membuat OpenCV membatasi rentang keluaran; di sini float dipertahankan untuk memeriksa nilai sebelum clipping.

Blur 5×5 dan sigma1 tetap untuk kedua k. Membesarkan k menguatkan detail sekaligus tekstur/noise dan berpotensi menghasilkan halo terang/gelap. Hasil terlihat lebih tajam belum membuktikan detail medis baru atau diagnosis lebih baik.

In [ ]:
# D1 : perbandingan adil, semua berangkat dari xray yang sama.
blur_xray = cv2.GaussianBlur(xray, GAUSS, SIGMA)
mask = xray - blur_xray
usm_float = {k: cv2.addWeighted(xray,1+k,blur_xray,-k,0) for k in K_VALUES}
usm = {k: to_u8(a) for k,a in usm_float.items()}
assert np.allclose(usm_float[2.0],xray+2.0*mask)
grid([xray,usm[1.0],usm[2.0]], ["Radiograf masukan", "USM k=1", "High-boost k=2"],limits=[(0,255)]*3,figsize=(11,4.0))
for k in K_VALUES: print(f"k={k:g}: clipping={clipping_pct(usm_float[k]):.4f}%")

In [ ]:
# D2 : ROI yang sama, skala intensitas sama; mask memakai skala bertanda.
grid([crop(xray,XRAY_ROI),crop(usm[1.0],XRAY_ROI),crop(usm[2.0],XRAY_ROI),crop(mask,XRAY_ROI)], ["ROI masukan", "USM k=1", "High-boost k=2", "Detail mask I−blur"],cmaps=["gray","gray","gray","coolwarm"],limits=[(0,255),(0,255),(0,255),(-30,30)],figsize=(12,3.5))
print("Perhatikan batas tulang, tekstur di dalamnya, dan garis terang/gelap di sisi batas.")
print("ROI jaringan nyata tidak diperlakukan sebagai ground truth noise.")
# Profil lokasi tetap: baris145, kolom370 sampai559; nilai sebelum clipping.
profile_y = 145
profile_x = np.arange(XRAY_ROI[0],XRAY_ROI[2])
with plt.rc_context({"font.size":15,"axes.titlesize":17,"axes.labelsize":15}):
    fig,axs=plt.subplots(1,2,figsize=(14,4.3))
    axs[0].imshow(crop(xray,XRAY_ROI),cmap="gray",vmin=0,vmax=255)
    axs[0].axhline(profile_y-XRAY_ROI[1],color="#DB5145",lw=2)
    axs[0].set_title("ROI batas tulang, baris y=145");axs[0].axis("off")
    for a,label,color in [(xray,"asli","#182845"),(usm_float[1.0],"k=1","#007F85"),(usm_float[2.0],"k=2","#C64738")]:
        axs[1].plot(profile_x,a[profile_y,profile_x],label=label,color=color,lw=1.2)
    axs[1].set(title="Profil intensitas sebelum clipping",xlabel="Kolom x",ylabel="Intensitas float")
    axs[1].legend();axs[1].grid(alpha=.2);fig.tight_layout();plt.show()
print("Garis profil sama untuk masukan, k1, k2; tinggi grafik bukan skor kualitas.")

In [ ]:
# D3 : eksperimen terkontrol: halo dan noise terlihat tanpa menebak anatomi.
step = np.full((64,256),40.0);step[:,128:]=200.0
step_blur=cv2.GaussianBlur(step,GAUSS,SIGMA)
step_k1=step+(step-step_blur);step_k2=step+2*(step-step_blur)
rng=np.random.default_rng(6)
flat=128+rng.normal(0,3,(128,128))
flat_blur=cv2.GaussianBlur(flat,GAUSS,SIGMA)
flat_k1=flat+(flat-flat_blur);flat_k2=flat+2*(flat-flat_blur)
noise_sd=[float(a[8:-8,8:-8].std()) for a in (flat,flat_k1,flat_k2)]
fig,axs=plt.subplots(1,2,figsize=(11,3.6))
for a,label,color in [(step,"masukan","#172035"),(step_k1,"k=1 sebelum clip","#0b9e94"),(step_k2,"k=2 sebelum clip","#ed6b35")]:axs[0].plot(np.arange(112,145),a[32,112:145],label=label,color=color)
axs[0].axhline(0,ls=":",color="gray");axs[0].axhline(255,ls=":",color="gray");axs[0].set(xlabel="Kolom x",ylabel="Intensitas float",title="Halo: overshoot / undershoot");axs[0].legend(fontsize=9);axs[0].grid(alpha=.2)
axs[1].bar(["masukan","k=1","k=2"],noise_sd,color=["#6d48d7","#0b9e94","#ed6b35"]);axs[1].set(ylabel="SD intensitas pada patch konstan",title="Noise terkontrol ikut dikuatkan")
for i,v in enumerate(noise_sd):axs[1].text(i,v+.1,f"{v:.2f}",ha="center")
fig.tight_layout();plt.show()
print("Noise seed6; dasar128, Gaussian sigma_noise3, ROI interior menghindari border.")
print("SD hanya ditafsirkan sebagai noise di patch sintetis ini, bukan jaringan asli.")

## E · Canny pada satelit: batas jalan dan pengecoh (10 menit)
Lima tahap konsep: smoothing Gaussian → gradien → non-maximum suppression → dua ambang → hysteresis. OpenCV `Canny` perlu diberi hasil **GaussianBlur eksplisit** bila ingin pra-penghalusan ini; kernel blur dan sigma dijaga sama dalam perbandingan.

`L2gradient=True` memakai magnitude Euclidean; default L1 memiliki skala berbeda, sehingga ambang perlu diperiksa kembali saat diganti. Tepi lemah dipertahankan jika terhubung langsung atau melalui rantai tepi lemah ke tepi kuat. Ambang 50/150 dan100/200 dipilih untuk citra 650×650 8-bit ini, bukan ketentuan rasio universal.

In [ ]:
# E1 : satu masukan pra-proses yang sama untuk kedua Canny.
sat_gray = cv2.cvtColor(sat_rgb,cv2.COLOR_RGB2GRAY)
sat_blur = cv2.GaussianBlur(sat_gray,GAUSS,SIGMA)
grid([sat_rgb,sat_gray,sat_blur],["RGB nyata yang telah disiapkan","Grayscale uint8","Gaussian5×5 sigma1"],limits=[None,(0,255),(0,255)],figsize=(11,3.6))

In [ ]:
# E2 : hanya dua ambang yang berubah.
edges = {pair:cv2.Canny(sat_blur,pair[0],pair[1],apertureSize=3,L2gradient=True) for pair in THRESHOLDS}
edge_fraction = {p:100*np.count_nonzero(e)/e.size for p,e in edges.items()}
grid([sat_rgb,edges[(50,150)],edges[(100,200)]],["Citra satelit masukan","Canny50/150","Canny100/200"],limits=[None,(0,255),(0,255)],figsize=(11,3.7))
for pair,pct in edge_fraction.items():print(f"Ambang{pair}: {pct:.4f}% piksel tepi")
print("Tepi lebih sedikit tidak otomatis lebih benar; baca juga garis yang hilang.")

In [ ]:
# E3 : overlay memperlihatkan Canny merespons jalan DAN atap/tekstur.
overlays={}
for pair,e in edges.items():
    overlay=sat_rgb.copy();overlay[e>0]=[255,73,156];overlays[pair]=overlay
grid([overlays[(50,150)],overlays[(100,200)]],["Overlay merah muda:50/150","Overlay merah muda:100/200"],figsize=(10,4.3))
print("Peta tepi belum membedakan kelas jalan, atap, kendaraan, bayangan, atau vegetasi.")

In [ ]:
# E4 : dua ROI tetap: jalan dan objek nonjalan.
arrays=[];titles=[]
for roi,tag in [(ROAD_ROI,"Jalan"),(ROOF_ROI,"Atap/tekstur")]:
    arrays.extend([crop(sat_rgb,roi),crop(edges[(50,150)],roi),crop(edges[(100,200)],roi)])
    titles.extend([f"{tag}:masukan",f"{tag}:50/150",f"{tag}:100/200"])
grid(arrays,titles,ncols=3,limits=[None,(0,255),(0,255)]*2,figsize=(10,6.2))
print("Cari dua bukti: batas jalan yang tetap terlihat dan tepi nonjalan yang ikut muncul.")
print("Label SpaceNet adalah centerline; keluaran Canny berupa batas. Tidak dihitung F1 antar-keduanya.")

## F · Cek hasil dan tulis pengamatan (8 menit)
Bagian wajib selesai setelah A–F. Simpan notebook yang sudah dijalankan dengan outputnya. Tulis 1–3 kalimat untuk masing-masing bukti di F2. Pakai posisi/ROI yang konkret; kata “lebih bagus” saja belum menjelaskan hasil.

**Rubrik100:** menjalankan dan mencatat parameter15, Sobel/Laplacian20, USM dan ROI25, Canny20, alasan pilihan serta batas klaim20. Empat metode tetap dikerjakan. Pengamatan berbeda diperbolehkan jika didukung keluaran yang terlihat.

In [ ]:
# F1 : cek bermakna: jenis angka, tanda, invariansi, dan masukan perbandingan.
constant=np.full((32,32),128.0)
constant_lap=cv2.Laplacian(constant,cv2.CV_64F,ksize=1)
assert np.allclose(constant_lap[2:-2,2:-2],0)
assert np.allclose(constant-LAMBDA*constant_lap,constant)
assert gx_card.min()<0<gx_card.max() and gy_card.min()<0<gy_card.max()
assert np.issubdtype(lap.dtype,np.floating)
assert np.allclose(mag,np.hypot(gx,gy))
assert all(e.shape==sat_gray.shape and set(np.unique(e)).issubset({0,255}) for e in edges.values())
assert usm[1.0].shape==usm[2.0].shape==xray_u8.shape
assert np.allclose(usm_float[1.0],xray+mask)
print("PASS: area konstan tidak diubah; turunan bertanda; rekonstruksi sesuai; keluaran konsisten.")
print("Ini cek numerik implementasi, bukan evaluasi diagnosis atau pemetaan jalan.")

In [ ]:
# F2 : ganti isi kutipan setelah melihat keluaran. Tidak perlu jawaban lisan.
pengamatan = {
    "Sobel": "Tulis batas vertikal/horizontal yang tampak di Gx/Gy dan arti magnitude.",
    "Laplacian": "Tulis perbedaan respons bertanda dengan citra I−0.25L; sebut ROI.",
    "USM": "Bandingkan k1 dan k2 di ROI sama; sebut detail dan efek sampingnya.",
    "Canny": "Bandingkan50/150 dan100/200; sebut batas jalan yang tampak serta tepi nonjalan.",
    "Pilihan": "Pilih parameter untuk salah satu tujuan dan jelaskan keterbatasan bukti."
}
for topik,jawaban in pengamatan.items():print(f"{topik}: {jawaban}\n")

## G · Eksperimen opsional setelah tugas wajib
G1–G2 tidak diperlukan untuk menyelesaikan 50 menit. Gunakan satu foto benda/dokumen sendiri, ubah **satu** parameter, lalu bandingkan dengan masukan yang sama. Contoh: sigma1→2 untuk USM, k1→1.5, atau ambang Canny. Pilih foto tanpa data pribadi; JPEG/PNG biasa, bukan DICOM.

G1 mencoba unggah file hanya jika berjalan di Colab. Jika unggah dibatalkan/tidak tersedia, tetap memakai satelit supplied sehingga Run all aman. G2 mengulang pipeline sederhana pada citra pilihan. Sharpening tidak memulihkan semua informasi akibat motion blur; untuk OCR, tidak ada klaim akurasi tanpa evaluasi OCR tersendiri.

In [ ]:
# G1 : opsional: default data supplied; set UPLOAD_OWN_IMAGE=True jika ingin unggah.
UPLOAD_OWN_IMAGE=False
own_gray=sat_gray.copy()
own_label="satelit supplied"
if UPLOAD_OWN_IMAGE:
    try:
        from google.colab import files
        uploaded=files.upload()
        if uploaded:
            name=next(iter(uploaded))
            decoded=cv2.imdecode(np.frombuffer(uploaded[name],dtype=np.uint8),cv2.IMREAD_GRAYSCALE)
            if decoded is None:raise ValueError("Gunakan JPEG/PNG yang bisa dibaca.")
            h,w=decoded.shape;scale=min(1.0,800/max(h,w))
            own_gray=cv2.resize(decoded,(max(1,round(w*scale)),max(1,round(h*scale))),interpolation=cv2.INTER_AREA)
            own_label=name
    except ImportError:
        print("Unggah Colab tidak tersedia di runtime ini; memakai data supplied.")
print("Citra eksperimen:",own_label,own_gray.shape)

In [ ]:
# G2 : ubah satu parameter, pertahankan lainnya.
EXPERIMENT_K=1.5
own_f=own_gray.astype(np.float64)
own_blur=cv2.GaussianBlur(own_f,GAUSS,SIGMA)
own_sharp=to_u8(own_f+EXPERIMENT_K*(own_f-own_blur))
own_edges=cv2.Canny(to_u8(own_blur),50,150,apertureSize=3,L2gradient=True)
grid([own_gray,own_sharp,own_edges],["Masukan yang sama",f"USM k={EXPERIMENT_K}","Canny50/150"],limits=[(0,255)]*3,figsize=(10,3.5))
print("Catat tujuan, satu parameter yang diubah, hasil terlihat, dan keterbatasannya.")

## Sumber untuk belajar lebih lanjut
- [OpenCV: Image Gradients](https://docs.opencv.org/4.x/d5/d0f/tutorial_py_gradients.html): Sobel, Laplacian, dan pentingnya output bertanda.
- [OpenCV: Laplace operator](https://docs.opencv.org/4.x/d5/db5/tutorial_laplace_operator.html): turunan kedua dan konvensi rekonstruksi.
- [scikit-image: Unsharp masking](https://scikit-image.org/docs/stable/auto_examples/filters/plot_unsharp_mask.html): hubungan radius/gain/detail.
- [OpenCV: Canny](https://docs.opencv.org/4.x/da/d5c/tutorial_canny_detector.html): contoh Gaussian pra-proses dan ambang.
- [OpenCV: Canny API](https://docs.opencv.org/4.x/dd/d1a/group__imgproc__feature.html): apertureSize dan L2gradient.
- [Canny1986](https://doi.org/10.1109/TPAMI.1986.4767851): good detection, localization, dan single response sebagai tujuan desain.
- PDF dosen Minggu6 (15 halaman, Endah Kristiani): kerangka Sub-CPMK dan studi kasus. Istilah kasus medis/GIS dibatasi di sini pada pengolahan visual yang benar-benar dikerjakan.

Materi ini menggunakan konvensi k pada slide dosen. Gambar SpaceNet yang diadaptasi (termasuk keluaran pengolahan) tetap berlisensi CC BY-SA4.0. Sumber citra dan transformasinya tersedia di `assets/subcpmk6/sources.json` dalam paket.